# ⚙️ Exercícios — Sistemas de Produção (Regras de Produção)

**Disciplina:** Inteligência Artificial | **Nível:** Intermediário

> Implemente encadeamento progressivo (forward chaining) e regressivo (backward chaining) com Python.


## 1. Encadeamento Progressivo (Forward Chaining)

In [ ]:
# Base de fatos inicial
fatos = {"chuva", "frio"}

# Regras no formato (condições, conclusão)
regras = [
    ({"chuva"},                    "molhado"),
    ({"frio", "molhado"},          "gripado"),
    ({"gripado"},                  "precisa_remedio"),
    ({"precisa_remedio"},          "vai_a_farmacia"),
    ({"frio"},                     "usa_casaco"),
    ({"chuva"},                    "usa_guarda_chuva"),
    ({"gripado", "vai_a_farmacia"},"toma_remedio"),
]

def forward_chaining(fatos_iniciais, regras):
    fatos = set(fatos_iniciais)
    print(f"Fatos iniciais: {sorted(fatos)}\n")
    iteracao = 0
    while True:
        novos = set()
        for condicoes, conclusao in regras:
            if condicoes.issubset(fatos) and conclusao not in fatos:
                novos.add(conclusao)
                print(f"  [{iteracao}] {condicoes} → {conclusao}")
        if not novos:
            break
        fatos |= novos
        iteracao += 1
    print(f"\nFatos finais: {sorted(fatos)}")
    return fatos

fatos_finais = forward_chaining(fatos, regras)


### 📝 Exercício 1

Adicione as seguintes regras e fatos ao sistema e execute novamente:
- Fato: `temperatura_alta`
- Regra: `temperatura_alta ∧ gripado → precisa_medico`
- Regra: `precisa_medico → telefona_clinica`

Verifique quais novos fatos são derivados.

In [ ]:
# ✏️ Adicione fatos e regras:
novos_fatos = {"chuva", "frio", "temperatura_alta"}
novas_regras = list(regras) + [
    # TODO: adicione as novas regras
]

forward_chaining(novos_fatos, novas_regras)


## 2. Encadeamento Regressivo (Backward Chaining)

Em vez de derivar todos os fatos, partimos do **objetivo** e tentamos prová-lo.

In [ ]:
# Regras para backward chaining (conclusao → condições)
regras_bc = {
    "molhado":         [{"chuva"}],
    "gripado":         [{"frio", "molhado"}],
    "precisa_remedio": [{"gripado"}],
    "vai_a_farmacia":  [{"precisa_remedio"}],
    "toma_remedio":    [{"gripado", "vai_a_farmacia"}],
    "usa_casaco":      [{"frio"}],
    "usa_guarda_chuva":[{"chuva"}],
}

def backward_chaining(objetivo, fatos, regras_bc, profundidade=0):
    indent = "  " * profundidade
    print(f"{indent}? Provar: {objetivo}")
    
    if objetivo in fatos:
        print(f"{indent}✅ {objetivo} é um FATO")
        return True
    
    if objetivo not in regras_bc:
        print(f"{indent}❌ {objetivo} não pode ser provado")
        return False
    
    for condicoes in regras_bc[objetivo]:
        print(f"{indent}Tentando regra: {condicoes} → {objetivo}")
        if all(backward_chaining(c, fatos, regras_bc, profundidade+1) for c in condicoes):
            print(f"{indent}✅ {objetivo} PROVADO!")
            return True
    
    print(f"{indent}❌ {objetivo} NÃO pôde ser provado")
    return False

fatos_bc = {"chuva", "frio"}
print("=== Backward Chaining: provar 'toma_remedio' ===")
resultado = backward_chaining("toma_remedio", fatos_bc, regras_bc)
print(f"\nResultado: {'PROVADO' if resultado else 'NÃO PROVADO'}")


### 📝 Exercício 2

Usando backward chaining, tente provar `usa_guarda_chuva` e `gripado` com os fatos `{"chuva"}` (sem `frio`). O que acontece? Por quê?

In [ ]:
fatos_sem_frio = {"chuva"}

print("=== Provar 'usa_guarda_chuva' ===")
backward_chaining("usa_guarda_chuva", fatos_sem_frio, regras_bc)

print("\n=== Provar 'gripado' ===")
backward_chaining("gripado", fatos_sem_frio, regras_bc)


## 3. Sistema de Produção Completo — Planeamento de Tarefas

In [ ]:
# Simulação de um sistema de produção para gestão de tarefas

class SistemaProducao:
    def __init__(self):
        self.memoria_trabalho = set()
        self.regras = []
        self.ciclos = 0
    
    def adicionar_fato(self, fato):
        self.memoria_trabalho.add(fato)
    
    def adicionar_regra(self, nome, condicoes, acoes):
        self.regras.append({"nome": nome, "cond": condicoes, "acoes": acoes})
    
    def ciclo_reconhece_age(self):
        """Um ciclo do loop reconhece-age."""
        regras_ativadas = [
            r for r in self.regras
            if r["cond"].issubset(self.memoria_trabalho)
            and not r["acoes"].issubset(self.memoria_trabalho)
        ]
        if not regras_ativadas:
            return False
        # Resolução de conflitos: escolhe a mais específica (mais condições)
        regra = max(regras_ativadas, key=lambda r: len(r["cond"]))
        print(f"  Ciclo {self.ciclos+1}: Disparando [{regra['nome']}]")
        self.memoria_trabalho |= regra["acoes"]
        self.ciclos += 1
        return True
    
    def executar(self, max_ciclos=20):
        print(f"Estado inicial: {sorted(self.memoria_trabalho)}")
        while self.ciclos < max_ciclos and self.ciclo_reconhece_age():
            pass
        print(f"\nEstado final ({self.ciclos} ciclos): {sorted(self.memoria_trabalho)}")

sp = SistemaProducao()
sp.adicionar_fato("ingredientes_disponiveis")
sp.adicionar_fato("fome")
sp.adicionar_regra("preparar_comida",   {"ingredientes_disponiveis","fome"}, {"comida_pronta"})
sp.adicionar_regra("comer",             {"comida_pronta","fome"},            {"saciado"})
sp.adicionar_regra("lavar_louça",       {"saciado"},                         {"cozinha_limpa"})
sp.adicionar_regra("descansar",         {"saciado","cozinha_limpa"},          {"descansado"})
sp.executar()


### 📝 Exercício Final

Crie um sistema de produção para um **processo educacional**:
- Fatos iniciais: `aluno_matriculado`, `sem_nota`
- Regras para: estudar → fazer_prova → receber_nota → aprovado/reprovado

Implemente e execute o sistema.

In [ ]:
# ✏️ Seu sistema de produção educacional:
sp2 = SistemaProducao()
# TODO: adicione fatos e regras
sp2.executar()
